# MMLACO-ARRW Reproducibility Notebook

This notebook documents the manuscript-aligned implementation of **MMLACO-ARRW** and the fixed-weight ablation variant **MMLACO-Fixed**.

- MMLACO-Fixed: alpha=0.9 and gamma=0.1 throughout the search.
- MMLACO-ARRW: alpha decreases linearly from 0.9 to 0.1, while gamma increases from 0.1 to 0.9 across 50 iterations.
- The adaptive weights affect candidate-subset evaluation and pheromone reinforcement, not the state-transition rule.

The benchmark datasets are third-party public resources and are not redistributed in this repository.

In [ ]:
import numpy as np
import pandas as pd
import random
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    hamming_loss, label_ranking_loss,
    label_ranking_average_precision_score,
    f1_score
)


In [ ]:
# Manuscript-aligned configuration
DATA_FILE = "scene.csv"
N_FEATURES = 294
N_LABELS = 6
BETA = 1
RHO = 0.1
N_ITER = 50
N_ANTS = 100
N_SELECTED = 40
Q0 = 0.6
INITIAL_PHEROMONE = 0.1
ALPHA_MAX = 0.9
ALPHA_MIN = 0.1
GAMMA_MIN = 0.1
GAMMA_MAX = 0.9
MLKNN_K = 10
TEST_SIZE = 0.30
SEED = 0


In [ ]:
def readdata(filename=DATA_FILE, n_features=N_FEATURES, n_labels=N_LABELS):
    aa = pd.read_csv(filename, header=None)
    data = aa.iloc[:, :n_features].copy()
    labels = aa.iloc[:, n_features:n_features+n_labels].copy()
    return data, labels, data.shape[0], data.shape[1]

data, labels, n, d = readdata()
print("Samples:", n, "| Features:", d, "| Labels:", labels.shape[1])

In [ ]:
from sklearn.metrics.cluster import normalized_mutual_info_score

def MI(data, labels):
    lsize = labels.shape[1]
    fsize = data.shape[1]
    mif = np.zeros((fsize, fsize), dtype=float)
    mil = np.zeros((fsize, lsize), dtype=float)
    for i in range(fsize):
        for j in range(fsize):
            mif[i, j] = normalized_mutual_info_score(data.iloc[:, i].values, data.iloc[:, j].values)
    for i in range(fsize):
        for j in range(lsize):
            mil[i, j] = normalized_mutual_info_score(data.iloc[:, i].values, labels.iloc[:, j].values)
    return mif, mil, lsize, fsize

mif, mil, lsize, fsize = MI(data, labels)

In [ ]:
def CreateGraph(mif):
    return 1.0 / np.maximum(mif, 1e-12)

def relevancy(mil):
    return np.sum(mil, axis=1)

def redandancy(f, M, mif):
    if len(M) == 0:
        return 0.0
    return float(np.sum(mif[f, M]))

def alpha_gamma(iteration, n_iter, mode="fixed"):
    if mode == "fixed":
        return ALPHA_MAX, GAMMA_MIN
    frac = iteration / float(max(n_iter - 1, 1))
    alpha_t = ALPHA_MAX - (ALPHA_MAX - ALPHA_MIN) * frac
    gamma_t = GAMMA_MIN + (GAMMA_MAX - GAMMA_MIN) * frac
    return alpha_t, gamma_t

def delta_tau(f, M, rel, mif, iteration, n_iter, mode):
    alpha_t, gamma_t = alpha_gamma(iteration, n_iter, mode)
    return alpha_t * rel[f] - gamma_t * redandancy(f, M, mif)

In [ ]:
def calculate_delta_ant(P, rel, mif, iteration, n_iter, mode):
    delta_ant = np.zeros(len(P), dtype=float)
    for k, M in enumerate(P):
        delta_ant[k] = sum(delta_tau(f, M, rel, mif, iteration, n_iter, mode) for f in M)
    return delta_ant

def calculate_delta_feature(P, delta_ant, fsize):
    delta_feature = np.zeros(fsize, dtype=float)
    for feature in range(fsize):
        for ant, path in enumerate(P):
            if feature in path:
                delta_feature[feature] += delta_ant[ant]
    return delta_feature

def notismember(fsize, M):
    used = set(M)
    return [i for i in range(fsize) if i not in used]

def RouletteWheel(P):
    total = np.sum(P)
    if total <= 0 or not np.isfinite(total):
        return np.random.randint(len(P))
    return np.random.choice(len(P), p=P/total)

def choosenextfeature(q, q0, N, Tau, G, io, beta):
    heuristic = np.array([max(0.0, Tau[dis] * (G[io, dis] ** beta)) for dis in N])
    if q <= q0:
        return N[int(np.argmax(heuristic))]
    return N[int(RouletteWheel(heuristic))]

In [ ]:
def MMLACO(data, beta, rho, n_iter, n_ants, n_selected, q0, initial_pheromone, mif, mil, mode="fixed", return_history=False, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
        random.seed(random_state)
    fsize = data.shape[1]
    G = CreateGraph(mif)
    rel = relevancy(mil)
    Tau = np.ones(fsize, dtype=float) * initial_pheromone
    history = []
    for iteration in range(n_iter):
        alpha_t, gamma_t = alpha_gamma(iteration, n_iter, mode)
        paths = [[] for _ in range(n_ants)]
        for k in range(n_ants):
            first = np.random.randint(0, fsize)
            paths[k].append(first)
            for _ in range(n_selected - 1):
                N = notismember(fsize, paths[k])
                nxt = choosenextfeature(random.random(), q0, N, Tau, G, paths[k][-1], beta)
                paths[k].append(nxt)
        delta_ant = calculate_delta_ant(paths, rel, mif, iteration, n_iter, mode)
        delta_feature = calculate_delta_feature(paths, delta_ant, fsize)
        Tau = (1.0 - rho) * Tau + delta_feature
        history.append({"iteration": iteration+1, "alpha": alpha_t, "gamma": gamma_t, "best_ant_score": float(np.max(delta_ant)), "mean_ant_score": float(np.mean(delta_ant)), "max_pheromone": float(np.max(Tau)), "mean_pheromone": float(np.mean(Tau))})
    selected = np.argsort(Tau)[::-1][:n_selected].astype(int)
    history = pd.DataFrame(history)
    return (selected, Tau, history) if return_history else selected

In [ ]:
SF_FIXED, Tau_FIXED, hist_FIXED = MMLACO(data, BETA, RHO, N_ITER, N_ANTS, N_SELECTED, Q0, INITIAL_PHEROMONE, mif, mil, mode="fixed", return_history=True, random_state=SEED)
SF_ARRW, Tau_ARRW, hist_ARRW = MMLACO(data, BETA, RHO, N_ITER, N_ANTS, N_SELECTED, Q0, INITIAL_PHEROMONE, mif, mil, mode="arrw", return_history=True, random_state=SEED)
print("Fixed:", SF_FIXED)
print("ARRW :", SF_ARRW)

In [ ]:
assert np.isclose(hist_ARRW.iloc[0]["alpha"], 0.9)
assert np.isclose(hist_ARRW.iloc[0]["gamma"], 0.1)
assert np.isclose(hist_ARRW.iloc[-1]["alpha"], 0.1)
assert np.isclose(hist_ARRW.iloc[-1]["gamma"], 0.9)
print(hist_ARRW[["iteration","alpha","gamma"]].head())
print(hist_ARRW[["iteration","alpha","gamma"]].tail())

In [ ]:
def evaluate(SF, data, labels, random_state=0, k=10):
    from skmultilearn.adapt import MLkNN
    SFData = data.iloc[:, np.asarray(SF, dtype=int)]
    X_train, X_test, Y_train, Y_test = train_test_split(SFData, labels, test_size=0.30, random_state=random_state)
    clf = MLkNN(k=k)
    clf.fit(X_train.values, Y_train.values)
    pred = clf.predict(X_test.values).toarray()
    true = Y_test.values
    return {"Hamming Loss": hamming_loss(true,pred), "Ranking Loss": label_ranking_loss(true,pred), "LRAP": label_ranking_average_precision_score(true,pred), "F1-micro": f1_score(true,pred,average="micro",zero_division=0), "F1-macro": f1_score(true,pred,average="macro",zero_division=0)}

print("MMLACO-Fixed:", evaluate(SF_FIXED, data, labels, random_state=SEED, k=MLKNN_K))
print("MMLACO-ARRW :", evaluate(SF_ARRW, data, labels, random_state=SEED, k=MLKNN_K))

## Notes for faithful reproduction

The manuscript reports ten independent runs for the main experiments. The notebook uses `SEED=0` as a single executable smoke/reproducibility example; use seeds 0–9 for ten independent runs when reproducing the reported protocol. For Flags, use the manuscript-specified eight-feature budget. The repository scripts provide repeated-run execution.

The notebook does not embed third-party benchmark datasets or fabricate the manuscript's reported numerical tables. It is intended to expose the method, ARRW schedule, fixed-vs-adaptive comparison, and evaluation protocol transparently.